In [2]:
# Install required Python packages into this notebook kernel
%pip install pandas numpy torch openpyxl

  Using cached pandas-2.3.3-cp310-cp310-win_amd64.whl (11.3 MB)
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl (12.9 MB)
  Using cached torch-2.10.0-cp310-cp310-win_amd64.whl (113.7 MB)
  Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)
  Using cached networkx-3.4.2-py3-none-any.whl (1.7 MB)
  Using cached fsspec-2026.2.0-py3-none-any.whl (202 kB)
  Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
  Using cached filelock-3.24.3-py3-none-any.whl (24 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
  Using cached markupsafe-3.0.3-cp310-cp310-win_amd64.whl (15 kB)
Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\arpit\OneDrive\Desktop\BioEdge_Project\.venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [1]:
import gzip
import os

print("--- BIOEDGE PHASE 1: CLINVAR DEEP SCANNER ---")

clinvar_file = 'variant_summary.txt.gz' 
mem_file = 'multi_gene_weights.mem'
target_genes = {"BRCA1": 0, "TP53": 1, "PTEN": 2, "APOE": 3}
MEMORY_SIZE = 512 
multi_gene_weights = [0] * MEMORY_SIZE

# Ensure we always have a value for count, even if scanning is skipped
count = 0

def resolve_file_path(file_name: str) -> str:
    """Search current and parent directories recursively for the given file name."""
    # Direct path first
    if os.path.exists(file_name):
        return file_name

    target = os.path.basename(file_name)
    search_roots = {os.getcwd(), os.path.abspath(os.path.join(os.getcwd(), '..'))}
    for root_start in search_roots:
        for root, _, files in os.walk(root_start):
            for f in files:
                if f == target:
                    return os.path.join(root, f)
    raise FileNotFoundError(f"Could not locate {file_name} starting from {os.getcwd()} or its parent")

# Try to locate the ClinVar file (e.g., in the data/ folder)
try:
    clinvar_path = resolve_file_path(clinvar_file)
    print(f"Scanning compressed file: {clinvar_path}...")
except FileNotFoundError as e:
    print(f"ERROR: {e}")
    clinvar_path = None

if clinvar_path is not None:
    try:
        # 'rt' mode allows us to read the zipped file as text directly
        with gzip.open(clinvar_path, 'rt', encoding='utf-8') as f:
            # Step 1: Find the Column Headers
            header_line = f.readline().strip()
            headers = header_line.split('\t')
            
            # ClinVar column names can vary, so we search for them
            try:
                idx_gene = headers.index('Symbol') if 'Symbol' in headers else headers.index('GeneSymbol')
                idx_sig  = headers.index('ClinicalSignificance')
                # We use Chromosome position (Start)
                idx_pos  = headers.index('Start') if 'Start' in headers else headers.index('PositionVCF')
                print(f"Found Columns -> Gene: {idx_gene}, Sig: {idx_sig}, Pos: {idx_pos}")
            except ValueError as e:
                print(f"Column Error: {e}. Check the README for column names.")
                clinvar_path = None
            
            # Step 2: Parse Data
            if clinvar_path is not None:
                for line in f:
                    cols = line.strip().split('\t')
                    if len(cols) <= max(idx_gene, idx_sig, idx_pos):
                        continue
                    
                    gene = cols[idx_gene].strip()
                    sig  = cols[idx_sig].lower()
                    
                    if gene in target_genes and "pathogenic" in sig:
                        gene_id = target_genes[gene]
                        try:
                            # Map the physical DNA position to our 128-word bank
                            pos_val = int(cols[idx_pos])
                            # This modulo maps the huge chromosome coordinate into our window
                            addr_offset = pos_val % 122 
                            final_addr = (gene_id * 128) + addr_offset
                            
                            multi_gene_weights[final_addr] = 64 # Hex 40
                            count += 1
                        except ValueError:
                            continue
                
                print(f"--- SUCCESS: Found {count} Clinical Hotspots! ---")
    except Exception as e:
        print(f"Critical Error: {e}")

# Step 3: Export to Hex for Vivado
with open(mem_file, 'w') as f:
    for weight in multi_gene_weights:
        f.write(f"{weight:02X}\n")

print(f"Generated {mem_file} with {count} hotspots.")

--- BIOEDGE PHASE 1: CLINVAR DEEP SCANNER ---
Scanning compressed file: c:\Users\arpit\OneDrive\Desktop\BioEdge_Project\data\variant_summary.txt.gz...
Found Columns -> Gene: 4, Sig: 6, Pos: 19
--- SUCCESS: Found 21058 Clinical Hotspots! ---
Generated multi_gene_weights.mem with 21058 hotspots.


In [2]:
# Copy this into a Python script to generate your full 512-row truth table
print("Address,SW3,SW2,Gene,DNA_In,Weight,Result")
for addr in range(512):
    sw3 = (addr >> 8) & 1
    sw2 = (addr >> 7) & 1
    gene = ["BRCA1", "TP53", "MYC", "Gene4"][addr // 128]
    # You can link this to your .mem file data
    print(f"{addr},{sw3},{sw2},{gene},1,ROM_DATA,VERDICT")

Address,SW3,SW2,Gene,DNA_In,Weight,Result
0,0,0,BRCA1,1,ROM_DATA,VERDICT
1,0,0,BRCA1,1,ROM_DATA,VERDICT
2,0,0,BRCA1,1,ROM_DATA,VERDICT
3,0,0,BRCA1,1,ROM_DATA,VERDICT
4,0,0,BRCA1,1,ROM_DATA,VERDICT
5,0,0,BRCA1,1,ROM_DATA,VERDICT
6,0,0,BRCA1,1,ROM_DATA,VERDICT
7,0,0,BRCA1,1,ROM_DATA,VERDICT
8,0,0,BRCA1,1,ROM_DATA,VERDICT
9,0,0,BRCA1,1,ROM_DATA,VERDICT
10,0,0,BRCA1,1,ROM_DATA,VERDICT
11,0,0,BRCA1,1,ROM_DATA,VERDICT
12,0,0,BRCA1,1,ROM_DATA,VERDICT
13,0,0,BRCA1,1,ROM_DATA,VERDICT
14,0,0,BRCA1,1,ROM_DATA,VERDICT
15,0,0,BRCA1,1,ROM_DATA,VERDICT
16,0,0,BRCA1,1,ROM_DATA,VERDICT
17,0,0,BRCA1,1,ROM_DATA,VERDICT
18,0,0,BRCA1,1,ROM_DATA,VERDICT
19,0,0,BRCA1,1,ROM_DATA,VERDICT
20,0,0,BRCA1,1,ROM_DATA,VERDICT
21,0,0,BRCA1,1,ROM_DATA,VERDICT
22,0,0,BRCA1,1,ROM_DATA,VERDICT
23,0,0,BRCA1,1,ROM_DATA,VERDICT
24,0,0,BRCA1,1,ROM_DATA,VERDICT
25,0,0,BRCA1,1,ROM_DATA,VERDICT
26,0,0,BRCA1,1,ROM_DATA,VERDICT
27,0,0,BRCA1,1,ROM_DATA,VERDICT
28,0,0,BRCA1,1,ROM_DATA,VERDICT
29,0,0,BRCA1,1,ROM_DATA,VERDICT
30,0,0,B